# Reconnaissance de Gestes avec CNN 1D

**Dataset**: 7 classes de gestes

**Architecture**: CNN 1D temporel pour temps réel

---
💡 **Activer le GPU**: Runtime → Change runtime type → GPU (T4)

## 1. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

## 2. Imports et Configuration GPU

In [ ]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 3. Configuration des Chemins

⚠️ **MODIFIER CES CHEMINS SELON TON GOOGLE DRIVE**

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/gesture_dataset'
MODEL_SAVE_PATH = '/content/drive/MyDrive/gesture_models'

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

if os.path.exists(DATASET_PATH):
    print(f'✅ Dataset trouvé: {DATASET_PATH}')
else:
    print(f'❌ Dataset non trouvé: {DATASET_PATH}')
    print('Modifier DATASET_PATH dans la cellule ci-dessus')

## 4. Classes de Gestes

In [ ]:
GESTURE_CLASSES = ['grab', 'idle', 'release', 'rotate_ccw', 'rotate_cw', 'zoom_in', 'zoom_out']
class_to_idx = {gesture: idx for idx, gesture in enumerate(GESTURE_CLASSES)}
idx_to_class = {idx: gesture for gesture, idx in class_to_idx.items()}

print(f'Classes ({len(GESTURE_CLASSES)}): {GESTURE_CLASSES}')

## 5. Chargement des Données

In [ ]:
def load_dataset(dataset_path, split='train'):
    X_list = []
    y_list = []
    split_path = os.path.join(dataset_path, split)
    
    for gesture_name in GESTURE_CLASSES:
        gesture_folder = os.path.join(split_path, gesture_name)
        if not os.path.exists(gesture_folder):
            continue
        
        npy_files = sorted([f for f in os.listdir(gesture_folder) if f.endswith('.npy')])
        print(f'{gesture_name}: {len(npy_files)} fichiers')
        
        for npy_file in npy_files:
            try:
                data = np.load(os.path.join(gesture_folder, npy_file))
                data_flat = data.reshape(data.shape[0], -1)
                X_list.append(data_flat)
                y_list.append(class_to_idx[gesture_name])
            except Exception as e:
                print(f'Erreur: {e}')
    
    return np.array(X_list), np.array(y_list)

print('Chargement train...')
X_train, y_train = load_dataset(DATASET_PATH, 'train')
print(f'\nChargement validation...')
X_val, y_val = load_dataset(DATASET_PATH, 'validation')

print(f'\n✅ Train: {X_train.shape}, Val: {X_val.shape}')

## 6. Normalisation

In [ ]:
def normalize_landmarks(X):
    X_norm = X.copy()
    n_samples, n_frames, n_features = X.shape
    
    for i in range(n_samples):
        landmarks = X[i].reshape(n_frames, 21, 3)
        wrist = landmarks[:, 0:1, :]
        landmarks_centered = landmarks - wrist
        palm_size = np.linalg.norm(landmarks[:, 9, :] - landmarks[:, 0, :], axis=1)
        palm_size = palm_size[:, np.newaxis, np.newaxis]
        palm_size = np.where(palm_size == 0, 1, palm_size)
        landmarks_norm = landmarks_centered / palm_size
        X_norm[i] = landmarks_norm.reshape(n_frames, -1)
    
    return X_norm

X_train_norm = normalize_landmarks(X_train)
X_val_norm = normalize_landmarks(X_val)
y_train_cat = to_categorical(y_train, len(GESTURE_CLASSES))
y_val_cat = to_categorical(y_val, len(GESTURE_CLASSES))

print('✅ Normalisation terminée')

## 7. Modèle CNN 1D

In [ ]:
def create_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv1D(64, 5, activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),
        
        layers.Conv1D(128, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),
        
        layers.Conv1D(256, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.4),
        
        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

input_shape = (X_train_norm.shape[1], X_train_norm.shape[2])
model = create_model(input_shape, len(GESTURE_CLASSES))
model.summary()

total_params = model.count_params()
print(f'\nParamètres: {total_params:,}')

## 8. Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_SAVE_PATH, 'best_model.keras'), 
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print('✅ Callbacks configurés')

## 9. Entraînement

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

history = model.fit(
    X_train_norm, y_train_cat,
    validation_data=(X_val_norm, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

## 10. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'Meilleure val_accuracy: {max(history.history["val_accuracy"]):.4f}')

## 11. Évaluation

In [ ]:
y_pred_proba = model.predict(X_val_norm)
y_pred = np.argmax(y_pred_proba, axis=1)
val_accuracy = accuracy_score(y_val, y_pred)

print(f'Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)')
print('\nRapport de classification:')
print(classification_report(y_val, y_pred, target_names=GESTURE_CLASSES, digits=4))

## 12. Matrice de Confusion

In [ ]:
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=GESTURE_CLASSES, yticklabels=GESTURE_CLASSES)
plt.xlabel('Prédiction')
plt.ylabel('Vérité')
plt.title('Matrice de Confusion')
plt.tight_layout()
plt.show()

## 13. Test de Performance

In [ ]:
import time

test_sample = X_val_norm[:32]
_ = model.predict(test_sample, verbose=0)

times = []
for _ in range(100):
    start = time.time()
    _ = model.predict(test_sample, verbose=0)
    times.append(time.time() - start)

avg_time_ms = np.mean(times) * 1000
avg_per_sample = avg_time_ms / len(test_sample)
fps = 1000 / avg_per_sample

print(f'Temps moyen: {avg_per_sample:.2f} ms/sample')
print(f'FPS théorique: {fps:.1f} FPS')
print(f'✅ Temps réel possible!' if avg_per_sample < 33 else '⚠️ Un peu lent')

## 14. Sauvegarde

In [ ]:
final_path = os.path.join(MODEL_SAVE_PATH, 'gesture_cnn1d_final.keras')
model.save(final_path)

import json
metadata = {
    'classes': GESTURE_CLASSES,
    'input_shape': list(input_shape),
    'val_accuracy': float(val_accuracy),
    'total_params': int(total_params)
}

with open(os.path.join(MODEL_SAVE_PATH, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ Modèle sauvegardé: {final_path}')
print(f'✅ Métadonnées sauvegardées')

## 15. Fonction d'Inférence

In [ ]:
def predict_gesture(landmarks_seq, model_obj, normalize=True):
    seq_flat = landmarks_seq.reshape(1, landmarks_seq.shape[0], -1)
    if normalize:
        seq_flat = normalize_landmarks(seq_flat)
    probs = model_obj.predict(seq_flat, verbose=0)[0]
    pred_idx = np.argmax(probs)
    return idx_to_class[pred_idx], probs[pred_idx], probs

test_seq = X_val[0].reshape(X_val[0].shape[0], 21, 3)
true_gesture = idx_to_class[y_val[0]]
pred_gesture, conf, probs = predict_gesture(test_seq, model)

print(f'Vérité: {true_gesture}')
print(f'Prédiction: {pred_gesture} (confiance: {conf:.2%})')

## 16. Résumé

In [ ]:
print('='*70)
print(' '*20 + 'RÉSUMÉ FINAL')
print('='*70)
print(f'\nArchitecture: Temporal CNN 1D')
print(f'Paramètres: {total_params:,}')
print(f'\nPerformance:')
print(f'  Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)')
print(f'  Temps inférence: {avg_per_sample:.2f} ms')
print(f'  FPS: {fps:.1f}')
print(f'\nFichiers sur Drive:')
print(f'  {MODEL_SAVE_PATH}/best_model.keras')
print(f'  {MODEL_SAVE_PATH}/gesture_cnn1d_final.keras')
print(f'  {MODEL_SAVE_PATH}/metadata.json')
print(f'\nClasses: {GESTURE_CLASSES}')
print('='*70)